# RNA11 with Cm spin-label: example data preparation

This example prepares the RNA11 with Cm spin-label data, writes the parameter arrays to text files, and makes the first twist-stretch plots.

The raw files omit rows for unpaired base pairs, but do not say which base-pair positions are missing. The `99.99` rows added below mark missing data only. Like in the old notebook, they are inserted after the frame marker and must not be interpreted as the actual positions of the unpaired base pairs. `----` values in the base-step and radius files are also changed to `99.99` in the processed files.

Run the notebook from the Analysis-Portfolio repository root. Raw and processed files are in `RNA_DNA_structural_data/MD_data/RNA11_wCm_tip3p/` and use the `..._RNA11_Cm_tip3p_...` filename pattern.

In [ ]:
import fileinput as fi
import numpy as np
import matplotlib.pyplot as plt
import dnaMD
from scipy.stats import linregress
import shutil

# dnaMD still uses the np.float alias removed in NumPy 2.
if not hasattr(np, 'float'):
    np.float = float

folder = 'RNA_DNA_structural_data/MD_data/RNA11_wCm_tip3p/'

## Add missing-row markers

This uses the same line-gap checks and `FileInput` replacements as the old notebook. One or two missing rows are added after the time marker. `----` values in the base-step and radius files are changed to `99.99`.

In [ ]:
for input_file, processed_file, one_missing_gap, two_missing_gap in [
    (folder + 'L-BP_RNA11_Cm_tip3p_raw.dat', folder + 'L-BP_RNA11_Cm_tip3p_processed.dat', 21, 20),
    (folder + 'L-BPH_RNA11_Cm_tip3p_raw.dat', folder + 'L-BPH_RNA11_Cm_tip3p_processed.dat', 20, 19),
    (folder + 'HelixRad_RNA11_Cm_tip3p_raw.dat', folder + 'HelixRad_RNA11_Cm_tip3p_processed.dat', 20, 19),
]:
    shutil.copyfile(input_file, processed_file)

    marker_text = '# Time ='
    marker_lines = np.array([])
    time_markers = []

    with open(processed_file) as myFile:
        for line_number, line in enumerate(myFile, 1):
            if marker_text in line:
                marker_lines = np.append(marker_lines, line_number)
                time_markers.append(line.rstrip('\n'))

    marker_gaps = np.diff(marker_lines)

    # Insert placeholders where a frame is one or two lines short.
    one_row_missing_frames = np.where(marker_gaps == one_missing_gap)
    one_row_missing_frames = one_row_missing_frames[0]

    for frame_index in one_row_missing_frames:
        if frame_index==0:
            search_marker = time_markers[frame_index]
            replace_marker = search_marker + '\n99.99   99.99   99.99   99.99   99.99   99.99'
        elif frame_index>0 and frame_index<10:
            search_marker = time_markers[frame_index]
            replace_marker = search_marker + '\n99.99   99.99   99.99   99.99   99.99   99.99'
        elif frame_index>9 and frame_index<100:
            search_marker = time_markers[frame_index]
            replace_marker = search_marker + '\n99.99   99.99   99.99   99.99   99.99   99.99'
        elif frame_index>99 and frame_index<1000:
            search_marker = time_markers[frame_index]
            replace_marker = search_marker + '\n99.99   99.99   99.99   99.99   99.99   99.99'
        elif frame_index>999 and frame_index<10000:
            search_marker = time_markers[frame_index]
            replace_marker = search_marker + '\n99.99   99.99   99.99   99.99   99.99   99.99'

        with fi.FileInput(processed_file, inplace=True) as File:
            for line in File:
                print(line.replace(search_marker, replace_marker), end='')

    two_rows_missing_frames = np.where(marker_gaps == two_missing_gap)
    two_rows_missing_frames = two_rows_missing_frames[0]

    for missing_frame_index in two_rows_missing_frames:
        if missing_frame_index==0:
            search_marker = time_markers[missing_frame_index]
            replace_marker = search_marker + '\n99.99   99.99   99.99   99.99   99.99   99.99\n99.99   99.99   99.99   99.99   99.99   99.99'
        elif missing_frame_index>0 and missing_frame_index<10:
            search_marker = time_markers[missing_frame_index]
            replace_marker = search_marker + '\n99.99   99.99   99.99   99.99   99.99   99.99\n99.99   99.99   99.99   99.99   99.99   99.99'
        elif missing_frame_index>9 and missing_frame_index<100:
            search_marker = time_markers[missing_frame_index]
            replace_marker = search_marker + '\n99.99   99.99   99.99   99.99   99.99   99.99\n99.99   99.99   99.99   99.99   99.99   99.99'
        elif missing_frame_index>99 and missing_frame_index<1000:
            search_marker = time_markers[missing_frame_index]
            replace_marker = search_marker + '\n99.99   99.99   99.99   99.99   99.99   99.99\n99.99   99.99   99.99   99.99   99.99   99.99'
        elif missing_frame_index>999 and missing_frame_index<10000:
            search_marker = time_markers[missing_frame_index]
            replace_marker = search_marker + '\n99.99   99.99   99.99   99.99   99.99   99.99\n99.99   99.99   99.99   99.99   99.99   99.99'

        with fi.FileInput(processed_file, inplace=True) as File:
            for line in File:
                print(line.replace(search_marker, replace_marker), end='')

    # Replace non-numeric values before dnaMD reads the files.
    with fi.FileInput(processed_file, inplace=True) as File:
        for line in File:
            print(line.replace('----', '99.99'), end='')

    total_frames = len(time_markers)

## Load parameters and save arrays

The `dnaMD` objects read the processed files. Parameter arrays are saved before the old `rm_neun` cleanup, so the text files keep the original frame sequence and its `99.99` markers.

In [ ]:
OL3Cm = dnaMD.DNA(20)
OL3Cm.set_base_step_parameters(
    folder + 'L-BPH_RNA11_Cm_tip3p_processed.dat',
    parameters='all', bp_step=[1, 19], step_range=True, helical=True
)

OL3Cmprop = dnaMD.DNA(20)
OL3Cmprop.set_base_pair_parameters(
    folder + 'L-BP_RNA11_Cm_tip3p_processed.dat',
    parameters='all', bp=[1, 20], bp_range=True
)

OL3Cmrad = dnaMD.DNA(20)
OL3Cmrad.set_helical_radius(
    folder + 'HelixRad_RNA11_Cm_tip3p_processed.dat',
    atomname='C1*', bp=[1, 19], bp_range=True
)

OL3Cmgroove = dnaMD.DNA(20)
groove_parameters = ['minor groove', 'minor groove refined',
                     'major groove', 'major groove refined']
OL3Cmgroove.set_major_minor_groove(
    folder + 'MGroove_RNA11_Cm_tip3p_processed.dat',
    bp_step=[1, 19], parameters=groove_parameters, step_range=True
)

rise, bp_idx = OL3Cm.get_parameters('h-rise', [1, 19], bp_range=True)
twist, bp_idx = OL3Cm.get_parameters('h-twist', [1, 19], bp_range=True)
inclination, bp_idx = OL3Cm.get_parameters('inclination', [1, 19], bp_range=True)
radius, bp_idx = OL3Cmrad.get_parameters('radius s-1', [1, 19], bp_range=True)
propeller, bp_idx = OL3Cmprop.get_parameters('propeller', [1, 20], bp_range=True)
groove_positions = {}
for parameter in groove_parameters:
    parameter_type = dnaMD.getParameterType(parameter)
    groove_positions[parameter] = [
        int(bp) for bp, values in OL3Cmgroove.data[parameter_type].items()
        if parameter in values
    ]

major, bp_idx = OL3Cmgroove.get_parameters(
    'major groove', groove_positions['major groove'], bp_range=False
)
major_refined, bp_idx = OL3Cmgroove.get_parameters(
    'major groove refined', groove_positions['major groove refined'], bp_range=False
)
minor, bp_idx = OL3Cmgroove.get_parameters(
    'minor groove', groove_positions['minor groove'], bp_range=False
)
minor_refined, bp_idx = OL3Cmgroove.get_parameters(
    'minor groove refined', groove_positions['minor groove refined'], bp_range=False
)

parameters = {
    'rise': rise,
    'twist': twist,
    'radius': radius,
    'inclination': inclination,
    'propeller': propeller,
    'major': major,
    'major_refined': major_refined,
    'minor': minor,
    'minor_refined': minor_refined,
}

for name, values in parameters.items():
    values = np.asarray(values)
    if values.ndim != 2 or values.shape[1] != total_frames:
        raise ValueError(f'{name} has shape {values.shape}; expected (positions, {total_frames}).')
    parameters[name] = values

for name, values in parameters.items():
    np.savetxt(folder + 'RNA11_Cm_tip3p_' + name + '.txt', values)
    print(f'Saved RNA11_Cm_tip3p_{name}.txt with shape {values.shape}')

## Remove marked frames and plot

This follows the old notebook: a time column is removed when one of its parameter values is exactly `99.99`, then the mean across base-pair positions is calculated. Large and negative values are left as they are.

In [ ]:
def rm_neun(parameter_values):
    parameter_values = np.asarray(parameter_values).copy()
    frames_to_remove = np.array([])
    for position_index in range(len(parameter_values)):
        missing_indices = np.where(parameter_values[position_index] == 99.99)
        frames_to_remove = np.append(frames_to_remove, missing_indices[0])

    frames_to_remove = np.sort(np.unique(frames_to_remove.astype(int)))
    parameter_values = np.delete(parameter_values, frames_to_remove, axis=1)
    return parameter_values

OL3Cmrise = rm_neun(parameters['rise']); avg_OL3Cmrise = np.mean(OL3Cmrise, axis=0)
OL3Cmtwist = rm_neun(parameters['twist']); avg_OL3Cmtwist = np.mean(OL3Cmtwist, axis=0)
OL3Cminc = rm_neun(parameters['inclination']); avg_OL3Cminc = np.mean(OL3Cminc, axis=0)
OL3Cmradius1 = rm_neun(parameters['radius']); avg_OL3Cmradius1 = np.mean(OL3Cmradius1, axis=0)
OL3Cmpropeller = rm_neun(parameters['propeller']); avg_OL3Cmpropeller = np.mean(OL3Cmpropeller, axis=0)
m_OL3Cm, b_OL3Cm, r_OL3Cm, p_OL3Cm, stderr_OL3Cm = linregress(avg_OL3Cmtwist, avg_OL3Cmrise)
print('OL3 Cm: ' + str(round(m_OL3Cm, 3)) + ' ; r-value: ' + str(round(r_OL3Cm, 2)))

cudof = 0
timeskip = 1
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(avg_OL3Cmtwist[np.arange(0, len(avg_OL3Cmtwist) - cudof, timeskip)],
         avg_OL3Cmrise[np.arange(0, len(avg_OL3Cmrise) - cudof, timeskip)], 'x')
plt.title('RNA11 with Cm')
plt.ylabel('helical rise / $\\AA$')
plt.xlabel('helical twist / $^\\circ$')
plt.subplot(1, 3, 2)
plt.plot(avg_OL3Cmtwist, avg_OL3Cmpropeller, 'x')
plt.ylabel('propeller / $^\\circ$')
plt.xlabel('helical twist / $^\\circ$')
plt.subplot(1, 3, 3)
plt.plot(avg_OL3Cmtwist, avg_OL3Cmradius1, 'x')
plt.ylabel('radius / $\\AA$')
plt.xlabel('helical twist / $^\\circ$')
plt.tight_layout()

fignom = 'RNA11_Cm_tip3p_twist_correlations.png'
#plt.savefig(fignom,dpi=1200)